In [ ]:
# read data
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")
df = pd.read_parquet("data.parquet")

Before modeling, I want to verify that the cleaned dataset is economically and statistically sensible. I'll check missingness, duplicates, data types, distributions, and whether important variables fall within plausible ranges.

Need to inspect data before and after cleaning. 

In [ ]:
# inspect
# Before modeling, I want to verify that the cleaned dataset is economically and statistically sensible. 
# I'll check missingness, duplicates, data types, distributions, and whether important variables fall within plausible ranges.

print(df.head())
print(df.shape)
print(df.dtypes)
print(df.columns)
print(df.info())
print(df.describe()) # numerical ranges

# check missing data
print(df.isna().sum())

# check duplicates
print(df.duplicated().sum())

# min max values
print(df["loan_amount"].min())
print(df["loan_amount"].max())

# categorical values
print(df["status"].value_counts())
print(df["category"].unique())

# check impossible values
print(df[df["loan_amount"] < 0])
print(df[df["income"] <= 0])
print(df[df["loan_ratio"] > 1])
print(np.isinf(df["col"]).sum())

In [ ]:
# clean

df = df.dropna()
df = df.dropna(subset=["income"]) # specific column

# fill with mean
df["x"] = df["x"].fillna(df["x"].mean())

# fill with median: when outliers exist
df["x"] = df["x"].fillna(df["x"].median())

# forward fill
df["x"] = df["x"].ffill()

# backwrd fill
df["x"] = df["x"].bfill()

# linear interpolation
df["x"] = df["x"].interpolate()

# time interpolation: if timestamps
df = df.set_index("date")
df["x"] = df["x"].interpolate(method="time")

#### Why forward fill?

If observations are ordered through time and I have reason to believe the most recent known value remains valid until the next observation, forward fill can be reasonable. But I wouldn't use it blindly because it can artificially create persistence.

#### Why interpolation instead of mean?"

Because if the data has an ordered or time-series structure, neighboring observations contain information about the missing value. Mean imputation ignores that structure and can flatten the variation in the data."

#### Why not interpolate everything?

Interpolation assumes some continuity between neighboring observations. For discrete categories or variables that can jump sharply, that assumption may not be valid.

In [ ]:
# logistic regression
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df.drop(columns=["default"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler() # check when to apply

X_train = scaler.fit_transform(X_train) # only on training data to avoid data leakage
X_test = scaler.transform(X_test) 

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(accuracy_score(y_test, pred)) # highly misleading if your dataset is imbalanced
print(confusion_matrix(y_test, pred))

print(classification_report(y_test, pred)) # gives Precision, Recall, and F1 score for every single class at once

In [ ]:
# using make_pipeline
from sklearn.pipeline import make_pipeline

# 1. Split data first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Build a pipeline that scales data, then applies Logistic Regression
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression() # Applies L2 regularization by default
)

# 3. Train the model (Fits the scaler and model safely on training data only)
pipeline.fit(X_train, y_train)

# 4. Evaluate (Transforms test data using the training scale automatically)
accuracy = pipeline.score(X_test, y_test)

y_pred = pipeline.predict(X_test)
pipe_prob = pipeline.predict_proba(X_test)[:, 1]
# then can use the same evaluation metrics


#### Which averaging method should you use for F1 score?

- average='macro': Calculates the F1 score for each class independently and takes their unweighted average. Use this if you care about minority classes equally, regardless of how rare they are.

- average='weighted': Calculates the F1 score for each class but weights them by how many samples belong to that class. Use this if you want to account for class imbalance based on true population sizes.

- average='micro': Globally counts total true positives, false negatives, and false positives. (Note: For standard multiclass classification, Micro F1 is mathematically identical to overall Accuracy).


In [ ]:
# probabilites 
prob = model.predict_proba(X_test)[:, 1]

# if 0.82, that means an estimated 82% probability of the positive class according to the fitted model.

In [ ]:
# accuracy 
from sklearn.metrics import precision_score, recall_score, roc_auc_score

print(precision_score(y_test, pred))
print(recall_score(y_test, pred))
print(roc_auc_score(y_test, prob))

In [ ]:
# confusion matrix
confusion_matrix(y_test, pred)

For defaults:
- TP = correctly predicted default
- FP = predicted default but borrower didn't default
- FN = borrower defaulted but model predicted no default
- TN = correctly predicted non-default
They may ask which error is worse.

Don't blindly answer.

Say:
"It depends on the business objective and cost of each error. In lending, missing a genuinely high-risk borrower can be especially costly, so false negatives may receive substantial attention."

In [ ]:
# outliers
Q1 = df["x"].quantile(0.25)
Q3 = df["x"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["x"] < lower) |
    (df["x"] > upper)
]